# Scripts to obtain results

The scripts are divided into three main parts according to the solver being used: 
- MUMPS
- SUPERLU_DIST
- PASTIX
- MKL PARDISO

Solver-specific options are set for all of these scripts.

If you want to use the `reorder()` function before calling the `solve_system()` function itself and "separate" the reordering this way, use the `-own_reordering true` option. Otherwise, use `-own_reordering false` and the reordering will be handled by the selected solver.

---

In [1]:
import os
import subprocess

In [2]:
# data paths
mat_input_path = "../matrices/bin/"
mat_permuted_path = "../matrices/bin/permuted/"
vec_input_path = "../matrices/bin/vec/"

# global paths for runs
EXE = "../petsc/arch-linux-c-opt/bin/mpirun -n {nproc} ../src/experiment"
MAT_INPUT_PATH = "../matrices/bin"
VEC_INPUT_PATH = "../matrices/bin/vec"

## MUMPS runs

In [11]:
# paths
LOG_DIR = "./mumps"  # this is where the log files will be saved
os.makedirs(LOG_DIR, exist_ok=True)

# parameters
nproc = 8          # number of processes being used
solver = "mumps"     
pc_type = "lu"
ordering = "parmetis" # set this option usually along with -own_reordering true,
                     # otherwise internal solver reordering will be used

mat_files = [f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin")]

for mat_file in mat_files:
    mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
    log_file = f"{ordering}_{os.path.splitext(mat_file)[0]}_{solver}_{nproc}procs.log"
    log_path = os.path.join(LOG_DIR, log_file)

    cmd = (
        f"{EXE.format(nproc=nproc)} "
        f"-mat_file {mat_path} "
        f"-pc_type {pc_type} "
        f"-mat_solver_type {solver} "
        #f"-own_reordering true "
        #f"-mat_ordering_type {ordering} "
        #f"-log_view "

        f"-mat_mumps_icntl_2 3 "          # output stream for diagnostic printing, statistics, and warning
        f"-mat_mumps_icntl_4 3 "          # level of printing (0 to 4)
        f"-mat_mumps_icntl_7 6 "          #  0=AMD, 2=AMF, 3=Scotch, 4=PORD, 5=Metis, 6=QAMD, and 7=auto
        f"-mat_mumps_icntl_28 2 "         # 1 for seq + ICNTL(7) ordering, 2 for par + ICNTL(29) ordering
        f"-mat_mumps_icntl_29 2 "         # parallel ordering 1 = ptscotch, 2 = parmetis
        #f"-mat_mumps_icntl_22 0 "         # out-of-core because I dont have enough memory :(
        f"-get_solution_norm true "
    )

    print(f"Running {mat_file} → {log_file}")
    with open(log_path, "w") as f:
        subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

print("======= COMPLETE =======")


Running rajat21.bin → parmetis_rajat21_mumps_8procs.log
======= COMPLETE =======


## SUPERLU_DIST runs

In [3]:
# paths
LOG_DIR = "./superlu_dist"
os.makedirs(LOG_DIR, exist_ok=True)

# parameters
nproc = 8            # number of processes being used
solver = "superlu_dist"     
pc_type = "lu"
ordering = "LargeDiag_AWPM__MMD_AT_PLUS_A" # set this option usually along with -own_reordering true,
                     # otherwise internal solver reordering will be used


mat_files = [f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin")]

for mat_file in mat_files:
    mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
    log_file = f"{ordering}_{os.path.splitext(mat_file)[0]}_{solver}_{nproc}procs.log"
    log_path = os.path.join(LOG_DIR, log_file)

    cmd = (
        f"{EXE.format(nproc=nproc)} "
        f"-mat_file {mat_path} "          # the file path to the matrix in .bin format
        f"-pc_type {pc_type} "            # the PC type
        f"-mat_solver_type {solver} "     # the solver to be used

        # f"-own_reordering true "         # true: enable reordering "separation", false: let solver handle reordering
        # f"-mat_ordering_type {ordering} " # the ordering to be used
        #f"-log_view "                     # print PETSc profiling
        
        f"-mat_superlu_dist_equil true "
        f"-mat_superlu_dist_iterrefine true "
        f"-mat_superlu_dist_replacetinypivot "

        f"-mat_superlu_dist_rowperm LargeDiag_AWPM " # <NOROWPERM,LargeDiag_MC64,LargeDiag_AWPM,MY_PERMR> - row permutation
        f"-mat_superlu_dist_colperm MMD_AT_PLUS_A " # <NATURAL,MMD_AT_PLUS_A,MMD_ATA,METIS_AT_PLUS_A,PARMETIS> - column
        f"-mat_superlu_dist_printstat "         # print factorization information

        f"-get_solution_norm true "
    )

    print(f"Running {mat_file} → {log_file}")
    with open(log_path, "w") as f:
        subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

print("======= COMPLETE =======")


Running rajat21.bin → LargeDiag_AWPM__MMD_AT_PLUS_A_rajat21_superlu_dist_8procs.log
======= COMPLETE =======


## PaStiX runs

In [16]:
# paths
LOG_DIR = "./pastix"
os.makedirs(LOG_DIR, exist_ok=True)

# parameters
nproc = 8            # number of processes being used
solver = "pastix"     
pc_type = "lu"
ordering = "metis" # set this option usually along with -own_reordering true,
                     # otherwise internal solver reordering will be used


mat_files = [f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin")]

for mat_file in mat_files:
    mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
    log_file = f"{ordering}_{os.path.splitext(mat_file)[0]}_{solver}_{nproc}procs.log"
    log_path = os.path.join(LOG_DIR, log_file)

    cmd = (
        f"{EXE.format(nproc=nproc)} "
        f"-mat_file {mat_path} "          # the file path to the matrix in .bin format
        f"-pc_type {pc_type} "            # the PC type
        f"-mat_solver_type {solver} "     # the solver to be used
        
        # f"-own_reordering true "         # true: enable reordering "separation", false: let solver handle reordering
        # f"-mat_ordering_type {ordering} " # the ordering to be used
        #f"-log_view "                     # print PETSc profiling

        f"-mat_pastix_verbose 1 "         # <0,1,2> - print level of information messages from PaStiX
        f"-mat_pastix_factorization 2 "     # <0,1,2,3,4> - Factorization algorithm (Cholesky, LDL^t, LU, LL^t, LDL^h)
        f"-mat_pastix_ordering 1 "          # <0,1> - Ordering (Scotch or Metis)
        f"-mat_pastix_thread_nbr 1 "        # Set the numbers of threads for each MPI process

        f"-get_solution_norm true "
    )

    print(f"Running {mat_file} → {log_file}")
    with open(log_path, "w") as f:
        subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

print("======= COMPLETE =======")

Running rajat21.bin → metis_rajat21_pastix_8procs.log
======= COMPLETE =======


## PARDISO runs

In [6]:
# paths
LOG_DIR = "./pardiso"
os.makedirs(LOG_DIR, exist_ok=True)

# parameters
nproc = 8            # number of processes being used
solver = "mkl_cpardiso"     
pc_type = "lu"
ordering = "external" # set this option usually along with -own_reordering true,
                     # otherwise internal solver reordering will be used


mat_files = [f for f in os.listdir(MAT_INPUT_PATH) if f.endswith(".bin")]

for mat_file in mat_files:
    mat_path = os.path.join(MAT_INPUT_PATH, mat_file)
    log_file = f"{ordering}_{os.path.splitext(mat_file)[0]}_{solver}_{nproc}procs.log"
    log_path = os.path.join(LOG_DIR, log_file)

    cmd = (
        f"{EXE.format(nproc=nproc)} "
        f"-mat_file {mat_path} "          # the file path to the matrix in .bin format
        f"-pc_type {pc_type} "            # the PC type
        f"-mat_solver_type {solver} "     # the solver to be used
        # f"-own_reordering true "         # true: enable reordering "separation", false: let solver handle reordering
        #f"-mat_ordering_type {ordering} " # the ordering to be used
        # f"-log_view "                     # print PETSc profiling

        #f"-mat_mkl_cpardiso_1 1 " 
        #f"-mat_mkl_cpardiso_65 2 "
        #f"-mat_mkl_cpardiso_68 1 "
        f"msglvl 1"

        f"-get_solution_norm true "
    )

    print(f"Running {mat_file} → {log_file}")
    with open(log_path, "w") as f:
        subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT)

print("======= COMPLETE =======")

Running bundle_adj.bin → external_bundle_adj_mkl_cpardiso_8procs.log
======= COMPLETE =======
